# T10 - 国债收益率波动应对研究

## 项目概述

本项目专注于分析国债收益率的波动特征，识别波动周期和影响因素，制定相应的投资策略和风险管理方案。

### 核心功能
1. **行情突然性分析** - 分析国债收益率变动的集中度特征
2. **90%行情集中度指标** - 衡量达到90%累计变动所需的时间占比
3. **策略回测与优化** - 规则策略与买入持有策略的对比分析
4. **年度对比分析** - 历史年度策略表现的系统性评估

### 数据源
- MySQL数据库: `bond.marketinfo_curve` 表
- 10年国债代码: L001619604
- 1年国债代码: L001618296

### 输出结果
- 行情阶段集中度分析图表
- 策略优化与回测结果
- 年度对比分析报告

---
## 1. 环境配置

导入必要的Python库并配置绘图参数。

In [ ]:
# 标准库
import os
import warnings
from datetime import datetime, timedelta

# 数据处理
import pandas as pd
import numpy as np

# 数据库
import sqlalchemy
from sqlalchemy.pool import NullPool

# 科学计算
from scipy.signal import find_peaks

# 可视化
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 配置
warnings.filterwarnings('ignore')

# 项目配置
from config import (
    BOND_CODES, DEFAULT_BOND_CODE, LOOKBACK_DAYS,
    OUTPUT_DIR, ASSETS_DIR, DATA_FILE_PATH,
    get_db_connection_string, get_stage_params, get_tenor,
    setup_plot_style
)

# 设置绘图风格
plt = setup_plot_style()

print("环境配置完成！")

---
## 2. 数据获取

连接数据库并获取国债收益率数据。

In [ ]:
def get_db_connection():
    """创建数据库连接"""
    conn_string = get_db_connection_string()
    sql_engine = sqlalchemy.create_engine(
        conn_string,
        poolclass=NullPool
    )
    return sql_engine.connect()


def fetch_yield_data_from_db(cursor, trade_code, start_date=None, end_date=None):
    """
    从数据库获取国债收益率数据
    
    参数:
        cursor: 数据库连接
        trade_code: 债券交易代码
        start_date: 起始日期
        end_date: 结束日期
    
    返回:
        DataFrame: 包含dt和CLOSE列的数据
    """
    query = f"""
    SELECT dt, CLOSE 
    FROM bond.marketinfo_curve 
    WHERE trade_code = '{trade_code}'
    """
    
    if start_date:
        query += f" AND dt >= '{start_date}'"
    if end_date:
        query += f" AND dt <= '{end_date}'"
    
    query += " ORDER BY dt"
    
    df = pd.read_sql(query, cursor)
    df['dt'] = pd.to_datetime(df['dt'])
    df['CLOSE'] = pd.to_numeric(df['CLOSE'], errors='coerce')
    df = df.dropna(subset=['CLOSE'])
    
    return df


def load_data_from_excel(file_path, trade_code):
    """
    从Excel文件加载数据（备用方法）
    
    参数:
        file_path: Excel文件路径
        trade_code: 债券交易代码
    
    返回:
        DataFrame: 处理后的数据
    """
    df = pd.read_excel(file_path)
    df_bond = df[df['trade_code'] == trade_code].copy()
    
    if df_bond.empty:
        return None
    
    df_bond['dt'] = pd.to_datetime(df_bond['dt'])
    df_bond = df_bond.sort_values(by='dt').set_index('dt')
    df_bond['CLOSE'] = pd.to_numeric(df_bond['CLOSE'], errors='coerce')
    df_bond = df_bond.dropna(subset=['CLOSE'])
    
    return df_bond


print("数据获取函数定义完成！")

In [ ]:
# 尝试从数据库获取数据
try:
    cursor = get_db_connection()
    print("数据库连接成功！")
    
    # 获取10年国债数据
    df_10y = fetch_yield_data_from_db(cursor, BOND_CODES['10年国债'])
    df_10y = df_10y.set_index('dt')
    print(f"10年国债数据: {len(df_10y)} 条记录")
    
    # 获取1年国债数据
    df_1y = fetch_yield_data_from_db(cursor, BOND_CODES['1年国债'])
    df_1y = df_1y.set_index('dt')
    print(f"1年国债数据: {len(df_1y)} 条记录")
    
    cursor.close()
    
except Exception as e:
    print(f"数据库连接失败: {e}")
    print("尝试从本地Excel文件加载数据...")
    
    # 备用：从Excel加载
    if os.path.exists(DATA_FILE_PATH):
        df_10y = load_data_from_excel(DATA_FILE_PATH, BOND_CODES['10年国债'])
        df_1y = load_data_from_excel(DATA_FILE_PATH, BOND_CODES['1年国债'])
        print("从Excel文件加载成功！")
    else:
        print("本地数据文件不存在，请检查数据源！")
        df_10y = pd.DataFrame()
        df_1y = pd.DataFrame()

---
## 3. 数据处理

计算每日收益率变化(BP)和预处理数据。

In [ ]:
def preprocess_bond_data(df):
    """
    预处理债券数据，计算每日变化
    
    参数:
        df: 原始数据DataFrame
    
    返回:
        DataFrame: 处理后的数据，包含change_bp列
    """
    df = df.copy()
    
    # 确保索引为日期
    if 'dt' in df.columns:
        df = df.set_index('dt')
    
    # 计算每日收益率变化 (BP)
    # 假设CLOSE是百分比数值，例如2.5代表2.5%
    # 0.01个百分点 = 1 BP
    df['change_bp'] = df['CLOSE'].diff() * 100
    df = df.dropna(subset=['change_bp'])
    
    return df


# 预处理数据
if not df_10y.empty:
    df_10y = preprocess_bond_data(df_10y)
    print(f"10年国债预处理完成: {len(df_10y)} 条有效记录")

if not df_1y.empty:
    df_1y = preprocess_bond_data(df_1y)
    print(f"1年国债预处理完成: {len(df_1y)} 条有效记录")

---
## 4. 核心逻辑

实现项目的核心分析功能。

### 4.1 90%行情集中度指标计算

In [ ]:
def calculate_90pct_concentration(df_period):
    """
    计算90%行情时间占比指标
    
    该指标衡量收益率变动的集中程度：
    - 分离正向和负向变化
    - 计算累计达到90%变动所需的天数占比
    - 数值越小表示行情突然性越强
    
    参数:
        df_period: 包含'CLOSE'列的DataFrame
    
    返回:
        float: 90%行情时间占比（0-1之间）
    """
    if len(df_period) < 2:
        return None
    
    df_period = df_period.copy()
    df_period['change_bp'] = df_period['CLOSE'].diff() * 100
    df_period = df_period.dropna(subset=['change_bp'])
    
    if len(df_period) == 0:
        return None
    
    changes_bp = df_period['change_bp'].values
    
    # 分离正值和负值变化的绝对值
    positive_changes_abs = np.abs(changes_bp[changes_bp > 0])
    negative_changes_abs = np.abs(changes_bp[changes_bp < 0])
    
    concentration_ratios = []
    
    # 计算正值变化的90%集中度
    if len(positive_changes_abs) > 0:
        positive_sorted = np.sort(positive_changes_abs)[::-1]
        positive_cumsum = np.cumsum(positive_sorted)
        total_positive = positive_cumsum[-1]
        
        if total_positive > 0:
            target_90pct = 0.9 * total_positive
            days_for_90pct = np.argmax(positive_cumsum >= target_90pct) + 1
            concentration_ratios.append(days_for_90pct / len(positive_sorted))
    
    # 计算负值变化的90%集中度
    if len(negative_changes_abs) > 0:
        negative_sorted = np.sort(negative_changes_abs)[::-1]
        negative_cumsum = np.cumsum(negative_sorted)
        total_negative = negative_cumsum[-1]
        
        if total_negative > 0:
            target_90pct = 0.9 * total_negative
            days_for_90pct = np.argmax(negative_cumsum >= target_90pct) + 1
            concentration_ratios.append(days_for_90pct / len(negative_sorted))
    
    if len(concentration_ratios) > 0:
        return np.mean(concentration_ratios) / 2
    return None


print("90%集中度指标计算函数定义完成！")

### 4.2 行情阶段划分与突然性分析

In [ ]:
def analyze_market_suddenness(df_bond, bond_name, output_dir=OUTPUT_DIR):
    """
    分析并可视化行情的突然性
    
    参数:
        df_bond: 债券数据DataFrame
        bond_name: 债券名称
        output_dir: 输出目录
    
    返回:
        DataFrame: 行情阶段集中度分析结果
    """
    if df_bond.empty or len(df_bond) < 30:
        print(f"{bond_name}: 数据不足，无法进行行情阶段分析")
        return None
    
    print(f"\n--- {bond_name} 行情突然性分析 ---")
    
    # 确保输出目录存在
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. 绘制收益率曲线
    plt.figure(figsize=(14, 7))
    plt.plot(df_bond.index, df_bond['CLOSE'], label=f'{bond_name} 收益率')
    plt.title(f'{bond_name} 国债收益率走势')
    plt.xlabel('日期')
    plt.ylabel('收益率 (%)')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, f'{bond_name}_yield_curve.png'))
    plt.close()
    print(f"已保存收益率走势图")
    
    # 2. 使用find_peaks识别行情阶段
    yield_series = df_bond['CLOSE']
    stage_params = get_stage_params(bond_name)
    
    prominence_value = stage_params['prominence_bp'] / 100  # 转换为百分比
    min_distance_days = stage_params['min_distance_days']
    
    print(f"阶段划分参数: prominence={prominence_value*100:.0f}BP, min_distance={min_distance_days}天")
    
    peak_indices, _ = find_peaks(yield_series, prominence=prominence_value, distance=min_distance_days)
    trough_indices, _ = find_peaks(-yield_series, prominence=prominence_value, distance=min_distance_days)
    
    critical_indices = sorted(list(set(list(peak_indices) + list(trough_indices))))
    
    # 添加首尾点
    if not critical_indices or critical_indices[0] != 0:
        critical_indices.insert(0, 0)
    if critical_indices[-1] != len(yield_series) - 1:
        critical_indices.append(len(yield_series) - 1)
    
    critical_indices = sorted(list(set(critical_indices)))
    
    # 3. 构建行情阶段
    stages = []
    for i in range(len(critical_indices) - 1):
        start_loc = critical_indices[i]
        end_loc = critical_indices[i + 1]
        
        if start_loc == end_loc:
            continue
        
        stage_slice = df_bond.iloc[start_loc:end_loc + 1]
        
        start_date = stage_slice.index[0]
        end_date = stage_slice.index[-1]
        start_price = stage_slice['CLOSE'].iloc[0]
        end_price = stage_slice['CLOSE'].iloc[-1]
        
        current_changes_bp = list(df_bond['change_bp'].iloc[start_loc:end_loc])
        total_stage_bp_change = (end_price - start_price) * 100
        stage_type = np.sign(total_stage_bp_change)
        
        if stage_type == 0:
            continue
        
        stages.append({
            'start_date': start_date,
            'end_date': end_date,
            'start_price': start_price,
            'end_price': end_price,
            'type': stage_type,
            'duration_days': (end_date - start_date).days + 1,
            'active_change_days': len([c for c in current_changes_bp if c != 0]),
            'changes_bp': current_changes_bp,
            'total_change_bp': total_stage_bp_change,
        })
    
    if not stages:
        print("未能识别出明确的行情阶段")
        return None
    
    stages_df = pd.DataFrame(stages)
    stages_df = stages_df[stages_df['active_change_days'] > 0]
    
    print(f"识别出 {len(stages_df)} 个行情阶段")
    
    # 4. 计算集中度
    concentration_results = []
    for idx, stage in stages_df.iterrows():
        if stage['active_change_days'] == 0 or not stage['changes_bp']:
            continue
        
        total_abs_change = abs(stage['total_change_bp'])
        if total_abs_change == 0:
            continue
        
        active_changes = [c for c in stage['changes_bp'] if c != 0]
        if not active_changes:
            continue
        
        current_cumulative = 0
        days_for_90_pct = -1
        
        for day_num, daily_change in enumerate(active_changes):
            current_cumulative += abs(daily_change)
            if days_for_90_pct == -1 and current_cumulative >= 0.9 * total_abs_change:
                days_for_90_pct = day_num + 1
                break
        
        num_active_days = len(active_changes)
        concentration_results.append({
            'start_date': stage['start_date'],
            'end_date': stage['end_date'],
            'type': '上涨' if stage['type'] == 1 else '下跌',
            'duration_days': stage['duration_days'],
            'active_days': num_active_days,
            'total_change_bp': total_abs_change,
            'days_for_90_pct': days_for_90_pct,
            'pct_time_for_90_pct': (days_for_90_pct / num_active_days * 100) if num_active_days > 0 and days_for_90_pct != -1 else None
        })
    
    concentration_df = pd.DataFrame(concentration_results)
    
    if not concentration_df.empty:
        # 保存结果
        concentration_df.to_csv(os.path.join(output_dir, f'{bond_name}_stage_concentration.csv'),
                                index=False, encoding='utf-8-sig')
        print(f"已保存行情阶段集中度分析")
        
        # 绘制90%行情时间分布
        valid_pct_times = concentration_df['pct_time_for_90_pct'].dropna()
        if not valid_pct_times.empty:
            plt.figure(figsize=(10, 6))
            plt.hist(valid_pct_times, bins=np.arange(0, 101, 5), edgecolor='black')
            plt.title(f'{bond_name}: 完成90%行情所需时间百分比分布')
            plt.xlabel('完成90%行情所用时间百分比 (%)')
            plt.ylabel('阶段数量')
            plt.xticks(np.arange(0, 101, 10))
            plt.grid(True)
            plt.savefig(os.path.join(output_dir, f'{bond_name}_90pct_change_time_distribution.png'))
            plt.close()
            
            # 输出统计结果
            avg_pct = valid_pct_times.mean()
            median_pct = valid_pct_times.median()
            print(f"平均: {avg_pct:.2f}%, 中位数: {median_pct:.2f}%")
            
            if median_pct < 30:
                print("结论: 行情具有明显突然性，大部分变动集中在较短时间")
            elif median_pct < 50:
                print("结论: 行情存在一定集中性")
            else:
                print("结论: 行情突然性不显著")
    
    return concentration_df


print("行情突然性分析函数定义完成！")

### 4.3 策略模拟与回测

In [ ]:
def simulate_trading_strategy(df_period_data, tenor, strategy_type, params=None):
    """
    模拟交易策略
    
    参数:
        df_period_data: 包含CLOSE和change_bp的DataFrame
        tenor: 债券久期（年）
        strategy_type: 'rule_based' 或 'buy_and_hold'
        params: 规则策略参数
            - up_thresh_bp: 上涨阈值
            - down_thresh_bp: 下跌阈值
            - buy_abs_increment: 买入增量
            - sell_abs_increment: 卖出减量
    
    返回:
        tuple: (详细数据DataFrame, 累计收益率%, 年化收益率%)
    """
    if df_period_data.empty or len(df_period_data) < 1:
        return pd.DataFrame(), 0.0, 0.0
    
    sim_df = df_period_data.copy()
    
    sim_df['position_at_eod'] = 0.0
    sim_df['position_at_sod'] = 0.0
    sim_df['avg_entry_yield'] = 0.0
    sim_df['pnl_daily_pct'] = 0.0
    
    current_pos = 0.0
    current_weighted_yield = 0.0
    
    for i in range(len(sim_df)):
        today_change_bp = sim_df['change_bp'].iloc[i]
        today_close = sim_df['CLOSE'].iloc[i]
        
        # 买入持有策略：首日建仓
        if i == 0 and strategy_type == 'buy_and_hold':
            pos_sod = 1.0
            avg_entry = today_close
            current_pos = 1.0
            current_weighted_yield = today_close
        else:
            pos_sod = current_pos
            avg_entry = current_weighted_yield / pos_sod if pos_sod > 1e-6 else 0.0
        
        sim_df.iloc[i, sim_df.columns.get_loc('position_at_sod')] = pos_sod
        sim_df.iloc[i, sim_df.columns.get_loc('avg_entry_yield')] = avg_entry
        
        # 计算每日收益
        price_pnl = pos_sod * tenor * (-today_change_bp / 100.0)
        coupon_pnl = pos_sod * (avg_entry / 100.0) / 365.25 if pos_sod > 1e-6 else 0.0
        daily_pnl = price_pnl + coupon_pnl
        sim_df.iloc[i, sim_df.columns.get_loc('pnl_daily_pct')] = daily_pnl
        
        # 更新仓位（规则策略）
        if strategy_type == 'rule_based' and params:
            if today_change_bp > params['up_thresh_bp']:  # 收益率上涨，买入
                buy_amount = min(params['buy_abs_increment'], 1.0 - current_pos)
                if buy_amount > 1e-6:
                    current_weighted_yield += buy_amount * today_close
                    current_pos += buy_amount
            elif today_change_bp < -params['down_thresh_bp']:  # 收益率下跌，卖出
                sell_amount = min(params['sell_abs_increment'], current_pos)
                if sell_amount > 1e-6 and current_pos > 1e-6:
                    current_weighted_yield -= (sell_amount / current_pos) * current_weighted_yield
                    current_pos -= sell_amount
                    if current_pos < 1e-6:
                        current_pos = 0.0
                        current_weighted_yield = 0.0
        
        sim_df.iloc[i, sim_df.columns.get_loc('position_at_eod')] = current_pos
    
    sim_df['cumulative_pnl_pct'] = sim_df['pnl_daily_pct'].cumsum()
    cumulative_pnl = sim_df['cumulative_pnl_pct'].iloc[-1]
    
    # 年化收益率
    start_date = sim_df.index.min()
    end_date = sim_df.index.max()
    num_days = (end_date - start_date).days + 1 if len(sim_df) > 1 else 1
    
    if num_days > 0:
        num_years = num_days / 365.25
        total_return_decimal = cumulative_pnl / 100.0
        if num_years > 1e-6 and total_return_decimal > -1:
            annualized_pnl = ((1 + total_return_decimal) ** (1 / num_years) - 1) * 100
        else:
            annualized_pnl = cumulative_pnl
    else:
        annualized_pnl = 0.0
    
    return sim_df, cumulative_pnl, annualized_pnl


print("策略模拟函数定义完成！")

In [ ]:
def run_strategy_optimization(df_bond, bond_name, tenor, output_dir=OUTPUT_DIR):
    """
    运行策略优化
    
    参数:
        df_bond: 债券数据
        bond_name: 债券名称
        tenor: 久期
        output_dir: 输出目录
    
    返回:
        dict: 优化结果
    """
    from config import STRATEGY_PARAMS
    
    print(f"\n--- {bond_name} 策略优化 ---")
    
    param_grid = STRATEGY_PARAMS
    
    best_params = None
    best_annual_pnl = -float('inf')
    all_results = []
    
    total_combos = (len(param_grid['up_thresh_bp_values']) * 
                   len(param_grid['down_thresh_bp_values']) *
                   len(param_grid['buy_increment_values']) *
                   len(param_grid['sell_increment_values']))
    
    print(f"遍历 {total_combos} 种参数组合...")
    
    count = 0
    for up_t in param_grid['up_thresh_bp_values']:
        for down_t in param_grid['down_thresh_bp_values']:
            for buy_incr in param_grid['buy_increment_values']:
                for sell_incr in param_grid['sell_increment_values']:
                    params = {
                        'up_thresh_bp': up_t,
                        'down_thresh_bp': down_t,
                        'buy_abs_increment': buy_incr,
                        'sell_abs_increment': sell_incr
                    }
                    count += 1
                    
                    _, _, annual_pnl = simulate_trading_strategy(
                        df_bond, tenor, 'rule_based', params
                    )
                    
                    result = params.copy()
                    result['annualized_pnl_pct'] = annual_pnl
                    all_results.append(result)
                    
                    if annual_pnl > best_annual_pnl:
                        best_annual_pnl = annual_pnl
                        best_params = params.copy()
    
    # 保存所有结果
    results_df = pd.DataFrame(all_results)
    results_df = results_df.sort_values('annualized_pnl_pct', ascending=False)
    results_df.to_csv(os.path.join(output_dir, f'{bond_name}_strategy_optimization.csv'),
                      index=False, encoding='utf-8-sig')
    
    # 计算买入持有基准
    _, _, bnh_annual_pnl = simulate_trading_strategy(df_bond, tenor, 'buy_and_hold')
    
    print(f"\n最佳规则策略参数: {best_params}")
    print(f"最佳规则策略年化收益: {best_annual_pnl:.2f}%")
    print(f"买入持有基准年化收益: {bnh_annual_pnl:.2f}%")
    
    if best_annual_pnl > bnh_annual_pnl:
        print("结论: 规则策略优于买入持有基准")
    else:
        print("结论: 买入持有基准优于或等于规则策略")
    
    return {
        'best_params': best_params,
        'best_annual_pnl': best_annual_pnl,
        'bnh_annual_pnl': bnh_annual_pnl,
        'all_results': results_df
    }


print("策略优化函数定义完成！")

---
## 5. 执行与测试

运行主分析流程并验证结果。

In [ ]:
# 确保输出目录存在
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 存储分析结果
analysis_results = {}

In [ ]:
# 分析10年国债
if not df_10y.empty:
    print("="*60)
    print("开始分析10年期国债")
    print("="*60)
    
    # 1. 行情突然性分析
    concentration_10y = analyze_market_suddenness(df_10y, "10年期国债")
    
    # 2. 策略优化
    optimization_10y = run_strategy_optimization(df_10y, "10年期国债", 10)
    
    analysis_results['10年国债'] = {
        'concentration': concentration_10y,
        'optimization': optimization_10y
    }
else:
    print("10年国债数据为空，跳过分析")

In [ ]:
# 分析1年国债
if not df_1y.empty:
    print("="*60)
    print("开始分析1年期国债")
    print("="*60)
    
    # 1. 行情突然性分析
    concentration_1y = analyze_market_suddenness(df_1y, "1年期国债")
    
    # 2. 策略优化
    optimization_1y = run_strategy_optimization(df_1y, "1年期国债", 1)
    
    analysis_results['1年国债'] = {
        'concentration': concentration_1y,
        'optimization': optimization_1y
    }
else:
    print("1年国债数据为空，跳过分析")

In [ ]:
# 汇总结果
print("\n" + "="*60)
print("分析结果汇总")
print("="*60)

for bond_name, results in analysis_results.items():
    print(f"\n{bond_name}:")
    
    if results.get('concentration') is not None:
        conc = results['concentration']
        if not conc.empty and 'pct_time_for_90_pct' in conc.columns:
            valid_times = conc['pct_time_for_90_pct'].dropna()
            if not valid_times.empty:
                print(f"  90%行情完成时间: 平均 {valid_times.mean():.1f}%, 中位数 {valid_times.median():.1f}%")
    
    if results.get('optimization'):
        opt = results['optimization']
        print(f"  最佳规则策略年化收益: {opt['best_annual_pnl']:.2f}%")
        print(f"  买入持有基准年化收益: {opt['bnh_annual_pnl']:.2f}%")

---
## 6. 工具函数

可复用的辅助工具函数。

In [ ]:
def calculate_var(returns, confidence=0.95):
    """
    计算风险价值(VaR)
    
    参数:
        returns: 收益率序列
        confidence: 置信水平
    
    返回:
        float: VaR值
    """
    return returns.quantile(1 - confidence)


def calculate_cvar(returns, confidence=0.95):
    """
    计算条件风险价值(CVaR)
    
    参数:
        returns: 收益率序列
        confidence: 置信水平
    
    返回:
        float: CVaR值
    """
    var = calculate_var(returns, confidence)
    return returns[returns <= var].mean()


def calculate_volatility(yield_data, window=20):
    """
    计算历史波动率
    
    参数:
        yield_data: 收益率数据
        window: 滚动窗口
    
    返回:
        Series: 年化波动率
    """
    returns = yield_data.pct_change()
    return returns.rolling(window).std() * np.sqrt(252)


def generate_date_list(start_date, end_date):
    """
    生成日期列表
    
    参数:
        start_date: 起始日期
        end_date: 结束日期
    
    返回:
        list: 日期列表
    """
    date_list = []
    current_date = start_date
    while current_date <= end_date:
        date_list.append(current_date)
        current_date += timedelta(days=1)
    return date_list


print("工具函数定义完成！")

In [ ]:
# 风险指标计算示例
if not df_10y.empty and 'pnl_daily_pct' in df_10y.columns:
    returns = df_10y['pnl_daily_pct']
    
    var_95 = calculate_var(returns, 0.95)
    cvar_95 = calculate_cvar(returns, 0.95)
    volatility = calculate_volatility(df_10y['CLOSE']).iloc[-1]
    
    print(f"\n10年国债风险指标:")
    print(f"  VaR(95%): {var_95:.4f}")
    print(f"  CVaR(95%): {cvar_95:.4f}")
    print(f"  年化波动率: {volatility:.4f}")

---
## 7. 总结

本项目实现了国债收益率波动应对研究的完整分析流程：

1. **数据获取**: 从数据库或Excel文件获取国债收益率数据
2. **数据处理**: 计算每日BP变化，预处理时间序列
3. **行情突然性分析**: 识别行情阶段，计算90%行情集中度
4. **策略回测**: 规则策略与买入持有策略的对比分析
5. **风险管理**: VaR、CVaR、波动率等风险指标计算

### 关键发现
- 行情变动具有集中性，大部分变动发生在较短时间内
- 规则策略通过参数优化可以在特定时期超越买入持有基准
- 风险管理指标有助于控制投资组合的下行风险